# KPI Engineering and Dashboard Preparation

This notebook transforms the synthetic event-level operations dataset into weekly KPI tables for operational monitoring and dashboard development.

The workflow converts raw instrument-run records into daily and weekly summaries that can be used to evaluate throughput, utilization, downtime, manual-entry burden, turnaround time, and quality outcomes.

In [1]:
# imports and paths
import pandas as pd
from pathlib import Path
from datetime import datetime

raw_file = Path("data/qc_instrument_usage.csv")
output_file = Path("data/qc_dashboard_ready.csv")

## Load Raw Event Data

The raw dataset contains one row per synthetic instrument run. Dates are parsed during import so time-based features can be created for daily and weekly aggregation.

In [2]:
# load and parse data
df = pd.read_csv(raw_file, parse_dates=["date"])

df.head()

,sample_id,instrument_id,instrument_type,date,test_type,duration_min,turnaround_hours,result,operator,product_family,manual_entry,automation_phase,downtime_flag,downtime_reason
0,S2024-00001,HPLC_01,HPLC,2024-01-01,Assay,35.5,29.5,pass,CF,Reagent_A,0,Before_Automation,0,NaN
1,S2024-00002,HPLC_01,HPLC,2024-01-01,Weigh,38.8,27.8,pass,CF,Reagent_A,0,Before_Automation,0,NaN
2,S2024-00003,HPLC_01,HPLC,2024-01-01,Assay,59.1,31.9,pass,DG,Media_X,0,Before_Automation,0,NaN
3,S2024-00004,HPLC_01,HPLC,2024-01-01,pH Check,31.9,35.9,pass,AD,Reagent_B,1,Before_Automation,0,NaN
4,S2024-00005,HPLC_01,HPLC,2024-01-01,Impurity,57.7,22.9,pass,CF,Media_X,0,Before_Automation,0,NaN


## Validate Required Fields

Defensive checks are included so the notebook can still run if optional fields are missing. Missing turnaround times are left as null values, while missing manual-entry flags are treated as zero.

In [3]:
# defensive guards
if "turnaround_hours" not in df.columns:
    df["turnaround_hours"] = pd.NA

if "manual_entry" not in df.columns:
    df["manual_entry"] = 0

df["manual_entry"] = df["manual_entry"].astype(int)

## Create Time and Automation Features

Calendar features support weekly KPI aggregation. The automation phase is assigned using the simulated July 1, 2024 implementation date.

In [4]:
# feature engineering
df["weekday"] = df["date"].dt.weekday
df["year"] = df["date"].dt.isocalendar().year
df["week"] = df["date"].dt.isocalendar().week

cutover = pd.Timestamp(2024, 7, 1)

df["automation_phase"] = df["date"].apply(
    lambda d: "Before_Automation" if d < cutover else "After_Automation"
)

df.head()

,sample_id,instrument_id,instrument_type,date,test_type,duration_min,turnaround_hours,result,operator,product_family,manual_entry,automation_phase,downtime_flag,downtime_reason,weekday,year,week
0,S2024-00001,HPLC_01,HPLC,2024-01-01,Assay,35.5,29.5,pass,CF,Reagent_A,0,Before_Automation,0,NaN,0,2024,1
1,S2024-00002,HPLC_01,HPLC,2024-01-01,Weigh,38.8,27.8,pass,CF,Reagent_A,0,Before_Automation,0,NaN,0,2024,1
2,S2024-00003,HPLC_01,HPLC,2024-01-01,Assay,59.1,31.9,pass,DG,Media_X,0,Before_Automation,0,NaN,0,2024,1
3,S2024-00004,HPLC_01,HPLC,2024-01-01,pH Check,31.9,35.9,pass,AD,Reagent_B,1,Before_Automation,0,NaN,0,2024,1
4,S2024-00005,HPLC_01,HPLC,2024-01-01,Impurity,57.7,22.9,pass,CF,Media_X,0,Before_Automation,0,NaN,0,2024,1


## Estimate Available Instrument Time

A simple operating-capacity model is used to estimate available instrument minutes by day.

Weekdays are assigned 8 hours of availability and weekends are assigned 4 hours. This supports utilization calculations at the weekly instrument level.

In [5]:
# available time
df["available_min"] = df["weekday"].map(
    lambda d: 480 if d < 5 else 240
)

## Daily Instrument Rollup

The event-level data is first aggregated to daily instrument-level summaries. This preserves day-level downtime flags before building weekly KPI tables.

In [6]:
# daily rollup
daily = (
    df.groupby(
        [
            "instrument_id",
            "instrument_type",
            "date",
            "year",
            "week",
            "automation_phase"
        ],
        as_index=False
    )
    .agg(
        runs=("test_type", "count"),
        total_minutes=("duration_min", "sum"),
        fails=("result", lambda s: (s == "fail").sum()),
        downtime_days=("downtime_flag", "max"),
        available_min=("available_min", "first"),
        runs_manual=("manual_entry", "sum"),
        avg_tat_hr=("turnaround_hours", "mean")
    )
)

daily["runs_auto"] = daily["runs"] - daily["runs_manual"]

daily.head()

,instrument_id,instrument_type,date,year,week,automation_phase,runs,total_minutes,fails,downtime_days,available_min,runs_manual,avg_tat_hr,runs_auto
0,BAL_01,Analytical_Balance,2024-01-01,2024,1,Before_Automation,5,241.7,0,0,480,1,29.420,4
1,BAL_01,Analytical_Balance,2024-01-02,2024,1,Before_Automation,5,210.9,1,0,480,1,28.460,4
2,BAL_01,Analytical_Balance,2024-01-03,2024,1,Before_Automation,8,378.2,0,0,480,5,28.550,3
3,BAL_01,Analytical_Balance,2024-01-04,2024,1,Before_Automation,8,333.5,1,0,480,2,31.050,6
4,BAL_01,Analytical_Balance,2024-01-05,2024,1,Before_Automation,4,162.0,0,0,480,0,30.825,4


## Weekly KPI Rollup

Daily summaries are aggregated to weekly instrument-level records for dashboard use. This produces one row per instrument per ISO week and automation phase.

In [7]:
# weekly rollup
weekly = (
    daily.groupby(
        [
            "instrument_id",
            "instrument_type",
            "year",
            "week",
            "automation_phase"
        ],
        as_index=False
    )
    .agg(
        runs=("runs", "sum"),
        runs_manual=("runs_manual", "sum"),
        runs_auto=("runs_auto", "sum"),
        total_minutes=("total_minutes", "sum"),
        fails=("fails", "sum"),
        downtime_days=("downtime_days", "sum"),
        available_min=("available_min", "sum"),
        avg_tat_hr=("avg_tat_hr", "mean")
    )
)

weekly.head()

,instrument_id,instrument_type,year,week,automation_phase,runs,runs_manual,runs_auto,total_minutes,fails,downtime_days,available_min,avg_tat_hr
0,BAL_01,Analytical_Balance,2024,1,Before_Automation,34,12,22,1484.3,2,0,2880,28.807857
1,BAL_01,Analytical_Balance,2024,2,Before_Automation,32,10,22,1368.4,1,1,2880,31.343554
2,BAL_01,Analytical_Balance,2024,3,Before_Automation,28,9,19,1253.3,3,0,2880,29.407398
3,BAL_01,Analytical_Balance,2024,4,Before_Automation,30,9,21,1441.2,1,0,2640,31.183333
4,BAL_01,Analytical_Balance,2024,5,Before_Automation,27,9,18,1231.4,1,0,2880,29.587381


## Calculate Operational KPIs

The weekly dataset includes rates and normalized measures used for monitoring operational performance.

Key metrics include failure rate, manual-entry rate, average run duration, utilization, and average turnaround time.

In [8]:
# KPI calculations
weekly["failure_rate_pct"] = (
    weekly["fails"] / weekly["runs"]
).fillna(0) * 100

weekly["manual_entry_rate_pct"] = (
    weekly["runs_manual"] / weekly["runs"]
).fillna(0) * 100

weekly["avg_run_duration_min"] = (
    weekly["total_minutes"] / weekly["runs"]
).fillna(0)

weekly["utilization_pct"] = (
    weekly["total_minutes"] / weekly["available_min"]
).clip(0, 1) * 100

weekly.head()

,instrument_id,instrument_type,year,week,automation_phase,runs,runs_manual,runs_auto,total_minutes,fails,downtime_days,available_min,avg_tat_hr,failure_rate_pct,manual_entry_rate_pct,avg_run_duration_min,utilization_pct
0,BAL_01,Analytical_Balance,2024,1,Before_Automation,34,12,22,1484.3,2,0,2880,28.807857,5.882353,35.294118,43.655882,51.538194
1,BAL_01,Analytical_Balance,2024,2,Before_Automation,32,10,22,1368.4,1,1,2880,31.343554,3.125000,31.250000,42.762500,47.513889
2,BAL_01,Analytical_Balance,2024,3,Before_Automation,28,9,19,1253.3,3,0,2880,29.407398,10.714286,32.142857,44.760714,43.517361
3,BAL_01,Analytical_Balance,2024,4,Before_Automation,30,9,21,1441.2,1,0,2640,31.183333,3.333333,30.000000,48.040000,54.590909
4,BAL_01,Analytical_Balance,2024,5,Before_Automation,27,9,18,1231.4,1,0,2880,29.587381,3.703704,33.333333,45.607407,42.756944


## Create Weekly Start Dates

A true Monday start date is created for each ISO year-week combination so the dashboard can plot weekly trends correctly.

In [9]:
# ISO week start date
def iso_week_start(year, week):
    return datetime.fromisocalendar(int(year), int(week), 1)

weekly["year_int"] = weekly["year"].astype(int)
weekly["week_int"] = weekly["week"].astype(int)

weekly["date"] = [
    iso_week_start(year, week)
    for year, week in zip(weekly["year_int"], weekly["week_int"])
]

weekly.drop(columns=["year_int", "week_int"], inplace=True)

weekly.head()

,instrument_id,instrument_type,year,week,automation_phase,runs,runs_manual,runs_auto,total_minutes,fails,downtime_days,available_min,avg_tat_hr,failure_rate_pct,manual_entry_rate_pct,avg_run_duration_min,utilization_pct,date
0,BAL_01,Analytical_Balance,2024,1,Before_Automation,34,12,22,1484.3,2,0,2880,28.807857,5.882353,35.294118,43.655882,51.538194,2024-01-01
1,BAL_01,Analytical_Balance,2024,2,Before_Automation,32,10,22,1368.4,1,1,2880,31.343554,3.125000,31.250000,42.762500,47.513889,2024-01-08
2,BAL_01,Analytical_Balance,2024,3,Before_Automation,28,9,19,1253.3,3,0,2880,29.407398,10.714286,32.142857,44.760714,43.517361,2024-01-15
3,BAL_01,Analytical_Balance,2024,4,Before_Automation,30,9,21,1441.2,1,0,2640,31.183333,3.333333,30.000000,48.040000,54.590909,2024-01-22
4,BAL_01,Analytical_Balance,2024,5,Before_Automation,27,9,18,1231.4,1,0,2880,29.587381,3.703704,33.333333,45.607407,42.756944,2024-01-29


## Export Dashboard-Ready Dataset

The final weekly KPI table is exported for Tableau dashboard development and downstream analysis.

In [10]:
weekly.sort_values(
    ["instrument_id", "year", "week"],
    inplace=True
)

output_file.parent.mkdir(parents=True, exist_ok=True)

weekly.to_csv(output_file, index=False)

print(f"Saved {len(weekly):,} rows to {output_file}")

weekly.head()

Saved 318 rows to data\qc_dashboard_ready.csv


,instrument_id,instrument_type,year,week,automation_phase,runs,runs_manual,runs_auto,total_minutes,fails,downtime_days,available_min,avg_tat_hr,failure_rate_pct,manual_entry_rate_pct,avg_run_duration_min,utilization_pct,date
0,BAL_01,Analytical_Balance,2024,1,Before_Automation,34,12,22,1484.3,2,0,2880,28.807857,5.882353,35.294118,43.655882,51.538194,2024-01-01
1,BAL_01,Analytical_Balance,2024,2,Before_Automation,32,10,22,1368.4,1,1,2880,31.343554,3.125000,31.250000,42.762500,47.513889,2024-01-08
2,BAL_01,Analytical_Balance,2024,3,Before_Automation,28,9,19,1253.3,3,0,2880,29.407398,10.714286,32.142857,44.760714,43.517361,2024-01-15
3,BAL_01,Analytical_Balance,2024,4,Before_Automation,30,9,21,1441.2,1,0,2640,31.183333,3.333333,30.000000,48.040000,54.590909,2024-01-22
4,BAL_01,Analytical_Balance,2024,5,Before_Automation,27,9,18,1231.4,1,0,2880,29.587381,3.703704,33.333333,45.607407,42.756944,2024-01-29
